In [ ]:
import os.path as osp
import os
import sys
sys.path.append('../')
import torch
from torch.utils.data import DataLoader
import numpy as np

from tqdm import tqdm

import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

import qdre
from evaluation_helpers import get_r_hats

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#DEVICE = 'cpu'
print(DEVICE)

/home/mdrnevich/test/QuasiDRE/SMEFT/../utils/__init__.py:4: UserWarning: The recommended fonts to use plothist were not found. You can install them by typing 'install_latin_modern_fonts' in your terminal. If it still does not work, please check the documentation at https://plothist.readthedocs.io/en/latest/usage/font_installation.html
  from . import plotting


cuda:0


In [2]:
DATA_DIR = "./data/"
SAVE_DIR = "./models/"

In [ ]:
source_file = osp.join(DATA_DIR, "SMEFT_SM_combined_tuple")
target_file = osp.join(DATA_DIR, "SMEFT_EFT_combined_tuple")

test_base_dataset = qdre.preprocessing.Dataset(source_file + "_test.npy", 0)
test_target_dataset = qdre.preprocessing.Dataset(target_file + "_test.npy", 1)

test_base_dataset.process()
test_target_dataset.process()

test_generator_data = qdre.preprocessing.CombinedDataset(test_base_dataset, test_target_dataset)
print(len(test_generator_data))

test_loader = DataLoader(test_generator_data, batch_size=int(2**10), shuffle=False)
test_nominal_loader = DataLoader(qdre.preprocessing.CombinedDataset(test_base_dataset), batch_size=int(2**10), shuffle=False)
test_target_loader = DataLoader(qdre.preprocessing.CombinedDataset(test_target_dataset), batch_size=int(2**10), shuffle=False)

477766


## Load the SMM model

In [ ]:
modpath = osp.join(SAVE_DIR, "classifier_SMM.zip")
SMM_model = qdre.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_SMM, weight_norm_SMM = qdre.preprocessing.load_scaling(modpath)
SMM_model.coefficients

(1.0, 1.2674115896224976)

In [5]:
print(SMM_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the $\textrm{SMM}_c$ PARE model

In [ ]:
batch_size = int(2**8)
modpath = osp.join(SAVE_DIR, "classifier_SMMc_batch{}.zip".format(batch_size))
pare_SMMc_model = qdre.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_pare_SMMc, weight_norm_pare_SMMc = qdre.preprocessing.load_scaling(modpath)
pare_SMMc_model.coefficients

(1.0, 1.267342209815979)

In [7]:
print(pare_SMMc_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the $\textrm{SMM}_c$ REVERT model

In [ ]:
batch_size = int(2**9)
modpath = osp.join(SAVE_DIR, "classifier_SMMc_revert_batch{}.zip".format(batch_size))
revert_SMMc_model = qdre.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_revert_SMMc, weight_norm_revert_SMMc = qdre.preprocessing.load_scaling(modpath)
revert_SMMc_model.coefficients

(1.0, 1.2443597316741943)

In [9]:
print(revert_SMMc_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the $\textrm{SMM}_r$ PARE model

In [ ]:
batch_size = int(2**8)
modpath = osp.join(SAVE_DIR, "classifier_SMMr_batch{}.zip".format(batch_size))
pare_SMMr_model = qdre.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_pare_SMMr, weight_norm_pare_SMMr = qdre.preprocessing.load_scaling(modpath)
pare_SMMr_model.coefficients

(1.0, 1.2675132751464844)

In [11]:
print(pare_SMMr_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the $\textrm{SMM}_r$ REVERT model

In [ ]:
batch_size = int(2**9)
modpath = osp.join(SAVE_DIR, "classifier_SMMr_revert_batch{}.zip".format(batch_size))
revert_SMMr_model = qdre.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_revert_SMMr, weight_norm_revert_SMMr = qdre.preprocessing.load_scaling(modpath)
revert_SMMr_model.coefficients

(1.0, 1.267569661140442)

In [13]:
print(revert_SMMr_model)

SingleMixtureClassifier(
  (subclassifiers): ModuleList(
    (0-1): 2 x Classifier(
      (model): Sequential(
        (0): Linear(in_features=16, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): ReLU()
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): ReLU()
        (6): Linear(in_features=128, out_features=1, bias=True)
        (7): Sigmoid()
      )
    )
  )
)


## Load the REVERT MLP classifier

In [ ]:
batch_size = int(2**8)
modpath = osp.join(SAVE_DIR, "classifier_revert_batch{}.zip".format(batch_size))
revert_model = qdre.models.load_model(modpath, device=DEVICE).to(DEVICE)
X_scaler_revert, weight_norm_revert = qdre.preprocessing.load_scaling(modpath)

In [15]:
print(revert_model)

Classifier(
  (model): Sequential(
    (0): Linear(in_features=16, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


## Evaluate the models

In [16]:
model_names = ["MLP REVERT", "RoSMM", "RoSMM-c PARE", "RoSMM-c REVERT", "RoSMM-r PARE", "RoSMM-r REVERT"]
models = [revert_model, SMM_model, pare_SMMc_model, revert_SMMc_model, pare_SMMr_model, revert_SMMr_model]
X_scalers = [X_scaler_revert, X_scaler_SMM, X_scaler_pare_SMMc, X_scaler_revert_SMMc, X_scaler_pare_SMMr, X_scaler_revert_SMMr]
weight_norms = [weight_norm_revert, weight_norm_SMM, weight_norm_pare_SMMc, weight_norm_revert_SMMc, weight_norm_pare_SMMr, weight_norm_revert_SMMr]

In [17]:
r_hats_list = []
for i in range(len(models)):
    loss = "revert" if "REVERT" in model_names[i] else "pare" if "PARE" in model_names[i] else "bce"
    mix = True if "SMM" in model_names[i] else False
    r_hat = get_r_hats(
        models[i],
        test_nominal_loader,
        X_scaler=X_scalers[i],
        weight_norm=weight_norms[i],
        loss=loss,
        mix=mix,
        leave=True,
        device=DEVICE
    )
    r_hats_list.append(r_hat)

100%|██████████| 234/234 [00:01<00:00, 136.68it/s]


# Compute the sliced wasserstein distance

In [18]:
def get_x(batch_list):
    x_batch_list = []
    w_batch_list = []
    for sample in batch_list:
        x_batch_list.append(sample[0])
        w_batch_list.append(sample[2])
    # print(x_batch_list[0].shape)
    x_batch = torch.stack(x_batch_list)
    w_batch = torch.stack(w_batch_list)
    # print(x_batch.shape, w_batch.shape)
    return x_batch, w_batch

In [19]:
@torch.no_grad()
def get_swd_data(loader):
    temp_x = []
    temp_w = []
    t = tqdm(enumerate(loader), total=len(loader))
    for i, batch in t:
        temp_x.append(batch[0])
        temp_w.append(batch[1])
        t.refresh()  # to show immediately the update
    return torch.cat(temp_x).numpy(), torch.cat(temp_w).numpy().flatten()

In [20]:
test_nominal_loader.collate_fn = lambda batch: get_x(batch)
test_target_loader.collate_fn = lambda batch: get_x(batch)

test_nominal_X = get_swd_data(test_nominal_loader)
test_target_X = get_swd_data(test_target_loader)

100%|██████████| 234/234 [00:00<00:00, 238.01it/s]


In [21]:
test_nominal_X = (test_nominal_X[0], test_nominal_X[1]/test_nominal_X[1].mean())
test_target_X = (test_target_X[0], test_target_X[1]/test_target_X[1].mean())

test_nominal_X[1].mean(), test_target_X[1].mean(), test_nominal_X[1].shape, test_target_X[1].shape

(np.float32(1.0), np.float32(0.9999996), (238883,), (238883,))

In [22]:
# n_subsamples = 200000
# index = np.random.choice(test_target_X[1].shape[0], size=n_subsamples, replace=False)

# test_target_X = (test_target_X[0][index], test_target_X[1][index])
# test_nominal_X[1].mean(), test_target_X[1].mean(), test_nominal_X[1].shape, test_target_X[1].shape

In [ ]:
N_PROJECTIONS = 50

print("Sliced Wasserstein distance between source and target: {}".format(qdre.metrics.extended_swd(test_nominal_X, test_target_X, n_projections=N_PROJECTIONS)))

for i, r_hat in enumerate(r_hats_list):
    ratios = test_nominal_X[1]*r_hat
    source_data = (test_nominal_X[0], ratios/np.sum(ratios))
    print("Sliced Wasserstein distance for model {}: {}".format(model_names[i], qdre.metrics.extended_swd(source_data, test_target_X, n_projections=N_PROJECTIONS)))

Sliced Wasserstein distance between source and target: 13.790769466500933
Sliced Wasserstein distance for model MLP REVERT: 0.2497859356192497
Sliced Wasserstein distance for model RoSMM: 1.4612147471630785
Sliced Wasserstein distance for model RoSMM-c PARE: 1.518375069598811
Sliced Wasserstein distance for model RoSMM-c REVERT: 0.8928211997112097
Sliced Wasserstein distance for model RoSMM-r PARE: 1.59161323621396
Sliced Wasserstein distance for model RoSMM-r REVERT: 0.5264863238566139


## Now look at computing the SWD many times and getting the mean and std

In [24]:
def get_SWD_dist(source_X, target_X, n_experiments=1000, n_projections=50):
    rvals = []
    for _ in tqdm(range(n_experiments)):
        rvals.append(utils.metrics.extended_swd(source_X, target_X, n_projections=n_projections))
    return rvals

In [25]:
SWD_SAVE_DIR = "./results_SWD/"
os.makedirs(SWD_SAVE_DIR, exist_ok=True)

In [26]:
## Get the reference distribution

# swd_dist = get_SWD_dist(test_nominal_X, test_target_X, n_projections=50)
# np.save(osp.join(SWD_SAVE_DIR, 'reference'), swd_dist)

In [ ]:
## Get the reweighting distributions

names = ["mlp_revert", "rosmm", "rosmmc_pare", "rosmmc_revert", "rosmmr_pare", "rosmmr_revert"]

start_idx = 0

for index in range(len(r_hats_list))[start_idx:]:
    r_hat = r_hats_list[index]
    r_hat /= r_hat.mean()
    
    source_data = (test_nominal_X[0], test_nominal_X[1]*r_hat)
    swd_dist = get_SWD_dist(source_data, test_target_X, n_projections=50)
    np.save(osp.join(SWD_SAVE_DIR, names[index]), swd_dist)

100%|██████████| 1000/1000 [1:36:23<00:00,  5.78s/it]


In [29]:
swd_dist = np.load(osp.join(SWD_SAVE_DIR, 'reference.npy'))
print("Sliced Wasserstein distance for reference:\t {:.3f} +/- {:.3f}".format(swd_dist.mean(), swd_dist.std()))

for index in range(len(r_hats_list)):
    swd_dist = np.load(osp.join(SWD_SAVE_DIR, names[index] + '.npy'))
    print("Sliced Wasserstein distance for model {}:\t {:.3f} +/- {:.3f}".format(model_names[index], swd_dist.mean(), swd_dist.std()))

Sliced Wasserstein distance for reference:	 13.388 +/- 1.344
Sliced Wasserstein distance for model MLP REVERT:	 0.276 +/- 0.020
Sliced Wasserstein distance for model RoSMM:	 1.423 +/- 0.129
Sliced Wasserstein distance for model RoSMM-c PARE:	 1.421 +/- 0.131
Sliced Wasserstein distance for model RoSMM-c REVERT:	 1.009 +/- 0.094
Sliced Wasserstein distance for model RoSMM-r PARE:	 1.674 +/- 0.153
Sliced Wasserstein distance for model RoSMM-r REVERT:	 0.494 +/- 0.044
